# Model initiation

In [1]:
from extraction_tools import *
from models import *
from glob import glob

save_dir = r'D:\AIC_BUFFER'

keyframe_feature_extraction_model = SigLIP2Model(name="google/siglip2-giant-opt-patch16-384", device="cuda")
keyframe_extractor = KeyframeAndFeatureExtractor(
    keyframe_feature_extraction_model,
    save_dir
)

# transcription_feature_extraction_model = BGEM3Model(name="BAAI/bge-m3", device="cuda")
# transcription_extraction_model = FasterWhisperModel(name="turbo", device="cuda")
# transcription_extractor = TranscriptAndFeatureProcessor(
#     transcription_extraction_model,
#     transcription_feature_extraction_model,
#     save_dir
# )


[INFO] Loading quantized google/siglip2-giant-opt-patch16-384
[INFO] google/siglip2-giant-opt-patch16-384 is on cuda device


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

# Extract keyframes

In [3]:
# vid_paths = glob(fr"D:\AIC26\vids\L25_V005.mp4")
vid_paths = [
    # News
    r"D:\AIC26\vids\L21_V022.mp4",
    r"D:\AIC26\vids\L22_V011.mp4",
    # Bike racing
    r"D:\AIC26\vids\L23_V007.mp4",
    r"D:\AIC26\vids\L23_V024.mp4",
    # Dragon dance
    r"D:\AIC26\vids\L24_V002.mp4",
    r"D:\AIC26\vids\L24_V004.mp4",
    # Lion dance
    r"D:\AIC26\vids\L24_V015.mp4",
    r"D:\AIC26\vids\L24_V038.mp4",
    # Online class
    r"D:\AIC26\vids\L25_V060.mp4",
    r"D:\AIC26\vids\L25_V050.mp4",
    # Cooking
    r"D:\AIC26\vids\L26_V496.mp4",
    r"D:\AIC26\vids\L26_V290.mp4",
    # Tradition
    r"D:\AIC26\vids\L28_V002.mp4",
    r"D:\AIC26\vids\L29_V018.mp4",
    # Socials
    r"D:\AIC26\vids\L30_V083.mp4",
    r"D:\AIC26\vids\L30_V015.mp4",
]

keyframe_extractor.extract_and_save_keyframes(
    vid_paths,
    feature_extraction_batch_size=16,
    clustering_similarity_threshold=0.98,
    keyframe_time_distance=3,
    max_workers=4
)

Extracting keyframes from video:   0%|          | 0/16 [00:00<?, ?it/s]

Extracting keyframes from video: 100%|██████████| 16/16 [04:28<00:00, 16.78s/it]


# Extract transcripts

In [ ]:
vid_paths = glob("E:/AIC_VID_ONLY/vids/*/L0*.mp4") + glob("E:/AIC_VID_ONLY/vids/*/L1*.mp4")
map_paths = glob("D:/AIC_NEW_EXTRACTION/*/L0*.csv") + glob("D:/AIC_NEW_EXTRACTION/*/L1*.csv")

if len(vid_paths) != len(map_paths): raise RuntimeError(f"Mismatch videos and maps count: {len(vid_paths)} to {len(map_paths)}")

transcription_extractor.extract_and_save_transcript(
    vid_paths,
    map_paths,
    transcription_batch_size=16,
    transcription_language="vi",
    alignment_max_time_distance=10,
    feature_extraction_batch_size=32,
    max_workers=4
)

# Size compaction

In [ ]:
from extraction_tools import FeaturePackager, InfoPackager

feat_compactor = FeaturePackager(r"D:\AIC26_NO_VIDS\feats", r"D:\AIC26_NO_VIDS\full_feats")
feat_compactor.compact_and_save_features(max_workers=8)


info_compactor = InfoPackager(r"D:\AIC26_NO_VIDS\maps", r"D:\AIC26_NO_VIDS\full_maps")
info_compactor.add_info_to_map("frame", r"D:\AIC26_NO_VIDS\trans", [("transcript", "transcript")], max_workers=8)
info_compactor.add_info_to_map("video", r"D:\AIC26_NO_VIDS\mets", [("watch_url", "base_yt_link")], max_workers=8)
info_compactor.save_map()

# Full run

In [ ]:
import torch
import gc
from extraction_tools import *
from models import *
from glob import glob

save_dir = r"D:\AIC26_NO_VIDS"
vid_paths = glob(r"D:\AIC26\vids\*.mp4")

keyframe_extractor = KeyframeAndFeatureExtractor(
	TransNetV2Model(device="cuda"),
    SigLIP2Model(name="google/siglip2-so400m-patch14-384", device="cuda"),
    BEiT3Model("timm/beit3_large_patch16_224.pt", device="cuda"),
    save_dir
)

# Customize the parameters to fit your hardware setup
keyframe_extractor.extract_and_save_keyframes(
    vid_paths,
    perceptive_feature_extraction_batch_size=16,
    fused_feature_extraction_batch_size=64,
    shot_confidence_threshold=0.5,
    clustering_similarity_threshold=0.92,
    max_workers=4
)

# If your VRAM is not plenty, you can delete the unused KeyframeAndFeatureExtractor after use
del keyframe_extractor
gc.collect()
torch.cuda.empty_cache()

map_paths = glob(r"D:\AIC26_NO_VIDS\maps\*.csv")

transcript_extractor = TranscriptAndFeatureProcessor(
    FasterWhisperModel(name="turbo", device="cuda"),
    BGEM3Model(name="BAAI/bge-m3", device="cuda"),
    save_dir
)

# Customize the parameters to fit your hardware setup
transcript_extractor.extract_and_save_transcript(
    vid_paths,
    map_paths,
    transcription_batch_size=16,
    transcription_language="vi",
    alignment_max_time_distance=10,
    feature_extraction_batch_size=32,
    max_workers=4
)

# If your VRAM is not plenty, you can delete the unused TranscriptAndFeatureProcessor after use
del transcript_extractor
gc.collect()
torch.cuda.empty_cache()

feat_compactor = FeaturePackager(r"D:\AIC26_NO_VIDS\feats", r"D:\AIC26_NO_VIDS\full_feats")
feat_compactor.compact_and_save_features(max_workers=8)

info_compactor = InfoPackager(r"D:\AIC26_NO_VIDS\maps", r"D:\AIC26_NO_VIDS\full_maps")
info_compactor.add_info_to_map("frame", r"D:\AIC26_NO_VIDS\trans", [("transcript", "transcript")], max_workers=8)
info_compactor.add_info_to_map("video", r"D:\AIC26_NO_VIDS\mets", [("watch_url", "base_yt_link")], max_workers=8)
info_compactor.save_map()

[INFO] Loading quantized turbo
[INFO] turbo is on cuda device
[INFO] Loading quantized BAAI/bge-m3
[INFO] BAAI/bge-m3 is on cuda device


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Extracting transcripts from video: 100%|██████████| 873/873 [1:31:03<00:00,  6.26s/it]


# Test

In [ ]:
import numpy as np
from glob import glob
from tqdm import tqdm

nan_count = 0
feats = glob(r'D:\AIC26_NO_VIDS\feats\**\*.npz', recursive=True)
for feat in tqdm(feats):
    feat = np.load(feat)
    nan_count += np.nan in feat["feat_siglip2_dense"]
    nan_count += np.nan in feat["feat_beit3_dense"]
    nan_count += np.nan in feat["feat_bgem3_dense"]
    nan_count += np.nan in feat["feat_bgem3_sparse_token_ids"]
    nan_count += np.nan in feat["feat_bgem3_sparse_weights"]
    print(feat["feat_siglip2_dense"])
    print(feat["feat_beit3_dense"])
    print(feat["feat_bgem3_dense"])
    print(feat["feat_bgem3_sparse_token_ids"])
    print(feat["feat_bgem3_sparse_weights"])

print(nan_count)

In [ ]:
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from extraction_tools import BEiT3Model

img_paths = [str(path) for path in list(Path(r"D:\AIC_NEW_EXTRACTION").glob("imgs/**/*.jpg"))]

beit3_model = BEiT3Model("timm/beit3_large_patch16_224.pt", device="cuda")
beit3_model.load()

In [ ]:
batch_size = 1024
feats = []
for i in tqdm(list(range(0, len(img_paths), batch_size))):
    j = i + batch_size if i + batch_size < len(img_paths) else len(img_paths)
    imgs = [Image.open(path) for path in img_paths[i:j]]
    feat = beit3_model.run(imgs, batch_size=batch_size)
    for f in feat: feats.append(f)

In [ ]:
import os
import numpy as np
feat_dir = Path(r"D:/AIC_NEW_EXTRACTION/feats/beit3_dense/")
for path, feat in zip(img_paths, feats):
    path = Path(path)
    if not os.path.exists(os.path.join(feat_dir, path.parts[-2])):
        os.mkdir(os.path.join(feat_dir, path.parts[-2]))
    np.save(str(Path(feat_dir) / path.parts[-2] / path.parts[-1]), feat)

In [ ]:
from glob import glob
import os
beit3_dense_paths = glob(r"D:\AIC26\feats\beit3_dense\*\*.npy")
for path in beit3_dense_paths:
    new_path = path.replace(".jpg", "")
    os.rename(path, new_path)